# 07. Build dataset C

Dataset C is made of prompts written one at a time by ChatGPT, following the professor's
instructions (realistic requests, varied wording, some overlap between tools, no templates).
It was produced in three parts:

| Part | File | Rows |
|---|---|---|
| pilot (reviewed before the full run) | `data_c/cybersecurity_pilot_30.csv` | 30 |
| batch 1 | `data_c/cybersecurity_new_550.csv` | 550 |
| batch 2 | `data_c/cybersecurity_extra_440.csv` | 440 |

A separate set of 405 prompts written by Claude (`data_c/cybersecurity_tool_selection_dedup.csv`,
one row per distinct request after removing near-copies) is kept apart as a **cross-source test
set**. It is never used for training, so it shows whether a classifier trained on ChatGPT's
wording still works on another model's wording.

This notebook merges the three ChatGPT parts, checks them, and writes `cyber_dataset_C.csv` and
`cyber_testset_claude405.csv`.

In [1]:
import re
from collections import Counter
import pandas as pd

TOOLS = ["SSHConnect", "NmapScan", "PortScan", "CheckVulnerability", "ReadAuthLog",
         "CheckFailedLogins", "ListeningPorts", "ListProcesses", "BlockIP",
         "GetSystemInfo", "ReadSyslog"]
PARTS = {"pilot": "data_c/cybersecurity_pilot_30.csv",
         "batch550": "data_c/cybersecurity_new_550.csv",
         "batch440": "data_c/cybersecurity_extra_440.csv"}

C = pd.concat([pd.read_csv(f)[["prompt", "label"]].assign(source=s) for s, f in PARTS.items()],
              ignore_index=True)
C = C.rename(columns={"label": "tool_ground_truth"})
C["row_id"] = [f"C{i:04d}" for i in range(len(C))]

T = pd.read_csv("data_c/cybersecurity_tool_selection_dedup.csv")[["prompt", "label"]]
T = T.rename(columns={"label": "tool_ground_truth"}).assign(source="claude")
T["row_id"] = [f"T{i:04d}" for i in range(len(T))]
print(len(C), "rows in dataset C |", len(T), "rows in the Claude test set")

1020 rows in dataset C | 405 rows in the Claude test set


## 1. Integrity checks

In [2]:
norm = lambda s: " ".join(re.sub(r"[^a-z0-9 ]", " ", str(s).lower()).split())

assert len(C) == 1020, len(C)
assert C.notna().all().all() and T.notna().all().all(), "missing values"
assert set(C.tool_ground_truth) == set(TOOLS) and set(T.tool_ground_truth) <= set(TOOLS)
assert not C.prompt.map(norm).duplicated().any(), "duplicate prompt in dataset C"
overlap = set(C.prompt.map(norm)) & set(T.prompt.map(norm))
assert not overlap, "a test-set prompt also appears in dataset C"
print("OK: 1020 rows, 11 valid labels, no missing values, no duplicates, no overlap with the test set")

OK: 1020 rows, 11 valid labels, no missing values, no duplicates, no overlap with the test set


## 2. Counts

In [3]:
pd.crosstab(C.tool_ground_truth, C.source, margins=True)

source,batch440,batch550,pilot,All
tool_ground_truth,,,,
BlockIP,40,50,3,93
CheckFailedLogins,40,50,3,93
CheckVulnerability,40,50,3,93
GetSystemInfo,40,50,2,92
ListProcesses,40,50,3,93
ListeningPorts,40,50,3,93
NmapScan,40,50,2,92
PortScan,40,50,3,93
ReadAuthLog,40,50,3,93


## 3. Wording checks

Length, the most common openings, and how often each tool's obvious cue word appears in its own prompts versus other tools' prompts.

In [4]:
words = C.prompt.map(lambda s: len(norm(s).split()))
print(f"words per prompt: median {int(words.median())}, range {words.min()}-{words.max()}, "
      f"{(words >= 30).mean():.0%} have 30 words or more")
print("most common 3-word openings:", Counter(" ".join(norm(p).split()[:3]) for p in C.prompt).most_common(5))
sents = C.prompt.map(lambda s: [x.strip().lower() for x in re.split(r"(?<=[.?!;])\s+", s) if len(x.split()) >= 4])
rep = Counter(x for ss in sents for x in set(ss))
print("sentences that appear in 3 or more prompts:", [(s, k) for s, k in rep.most_common(5) if k >= 3])

CUE = {"NmapScan": r"nmap", "SSHConnect": r"\bssh\b", "ReadSyslog": r"syslog", "BlockIP": r"\bblock",
       "ListProcesses": r"process", "CheckVulnerability": r"vulnerab|\bcve", "CheckFailedLogins": r"fail",
       "ReadAuthLog": r"auth", "PortScan": r"\bports?\b", "ListeningPorts": r"listen",
       "GetSystemInfo": r"kernel|hostname|uptime"}
cue = pd.DataFrame({"cue": CUE,
                    "in own prompts": {k: C[C.tool_ground_truth == k].prompt.str.contains(p, case=False).mean() for k, p in CUE.items()},
                    "in other prompts": {k: C[C.tool_ground_truth != k].prompt.str.contains(p, case=False).mean() for k, p in CUE.items()}})
cue[["in own prompts", "in other prompts"]] = (cue[["in own prompts", "in other prompts"]] * 100).round(0)
cue

words per prompt: median 23, range 5-63, 37% have 30 words or more
most common 3-word openings: [('i need to', 14), ('i need the', 9), ('can you check', 6), ('can you identify', 5), ('i don t', 5)]
sentences that appear in 3 or more prompts: []


,cue,in own prompts,in other prompts
NmapScan,nmap,0.0,0.0
SSHConnect,\bssh\b,7.0,3.0
ReadSyslog,syslog,11.0,0.0
BlockIP,\bblock,19.0,1.0
ListProcesses,process,55.0,3.0
CheckVulnerability,vulnerab|\bcve,44.0,1.0
CheckFailedLogins,fail,37.0,4.0
ReadAuthLog,auth,59.0,7.0
PortScan,\bports?\b,73.0,10.0
ListeningPorts,listen,47.0,1.0


## 4. Save

In [5]:
C[["prompt", "tool_ground_truth", "source", "row_id"]].to_csv("cyber_dataset_C.csv", index=False)
T[["prompt", "tool_ground_truth", "source", "row_id"]].to_csv("cyber_testset_claude405.csv", index=False)
print("saved cyber_dataset_C.csv and cyber_testset_claude405.csv")

saved cyber_dataset_C.csv and cyber_testset_claude405.csv
